# Extract Custom Fields from Your File

This notebook demonstrates how to use analyzers to extract custom fields from your input files.

## Prerequisites
1. Ensure Azure AI service is configured following [steps](../README.md#configure-azure-ai-service-resource)
2. Install the required packages to run the sample.

In [8]:
#%pip install -r ../requirements.txt

## Analyzer Templates

Below is a collection of analyzer templates designed to extract fields from various input file types.

These templates are highly customizable, allowing you to modify them to suit your specific needs. For additional verified templates from Microsoft, please visit [here](../analyzer_templates/README.md).

In [9]:
extraction_templates = {
    "invoice":            ('../analyzer_templates/invoice.json',         '../data/invoice.pdf'            ),
    "chart":              ('../analyzer_templates/image_chart.json',     '../data/pieChart.jpg'           ),
    "call_recording":     ('../analyzer_templates/call_recording_analytics.json', '../data/callCenterRecording.mp3'),
    "conversation_audio": ('../analyzer_templates/conversational_audio_analytics.json', '../data/callCenterRecording.mp3'),
    "marketing_video":    ('../analyzer_templates/marketing_video.json', '../data/FlightSimulator.mp4'              )
}

Specify the analyzer template you want to use and provide a name for the analyzer to be created based on the template.

In [10]:
import uuid

ANALYZER_TEMPLATE = "invoice"
ANALYZER_ID = "field-extraction-sample-" + str(uuid.uuid4())

(analyzer_template_path, analyzer_sample_file_path) = extraction_templates[ANALYZER_TEMPLATE]

## Create Azure AI Content Understanding Client

> The [AzureContentUnderstandingClient](../python/content_understanding_client.py) is a utility class containing functions to interact with the Content Understanding API. Before the official release of the Content Understanding SDK, it can be regarded as a lightweight SDK.


In [ ]:
import logging
import json
import os
import sys
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

load_dotenv(find_dotenv())
logging.basicConfig(level=logging.INFO)

AZURE_AI_ENDPOINT = os.getenv("AZURE_AI_ENDPOINT")
AZURE_AI_API_VERSION = os.getenv("AZURE_AI_API_VERSION", "2024-12-01-preview")

# Add the parent directory to the path to use shared modules
parent_dir = Path(Path.cwd()).parent
sys.path.append(str(parent_dir))
from python.content_understanding_client import AzureContentUnderstandingClient

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

client = AzureContentUnderstandingClient(
    endpoint=AZURE_AI_ENDPOINT,
    api_version=AZURE_AI_API_VERSION,
    token_provider=token_provider,
    x_ms_useragent="azure-ai-content-understanding-python/field_extraction", # This header is used for sample usage telemetry, please comment out this line if you want to opt out.
)

## Create Analyzer from the Template

In [12]:
response = client.begin_create_analyzer(ANALYZER_ID, analyzer_template_path=analyzer_template_path)
result = client.poll_result(response)

print(json.dumps(result, indent=2))

INFO:python.content_understanding_client:Analyzer field-extraction-sample-df9a6007-a6a9-4b4c-999b-8f0d8b70298b create request accepted.
INFO:python.content_understanding_client:Request 42ef58e3-46e3-4c46-84c3-a172ec478478 in progress ...
INFO:python.content_understanding_client:Request 42ef58e3-46e3-4c46-84c3-a172ec478478 in progress ...
INFO:python.content_understanding_client:Request result is ready after 4.92 seconds.


{
  "id": "42ef58e3-46e3-4c46-84c3-a172ec478478",
  "status": "Succeeded",
  "result": {
    "analyzerId": "field-extraction-sample-df9a6007-a6a9-4b4c-999b-8f0d8b70298b",
    "description": "Sample invoice analyzer",
    "createdAt": "2025-03-06T17:00:47Z",
    "lastModifiedAt": "2025-03-06T17:00:50Z",
    "config": {
      "returnDetails": false,
      "enableOcr": true,
      "enableLayout": true,
      "enableBarcode": false,
      "enableFormula": false,
      "disableContentFiltering": false
    },
    "fieldSchema": {
      "fields": {
        "VendorName": {
          "type": "string",
          "method": "extract",
          "description": "Vendor issuing the invoice"
        },
        "Items": {
          "type": "array",
          "method": "extract",
          "items": {
            "type": "object",
            "properties": {
              "Description": {
                "type": "string",
                "method": "extract",
                "description": "Description of

## Extract Fields Using the Analyzer

After the analyzer is successfully created, we can use it to analyze our input files.

In [13]:
response = client.begin_analyze(ANALYZER_ID, file_location=analyzer_sample_file_path)
result = client.poll_result(response)

print(json.dumps(result, indent=2))

INFO:python.content_understanding_client:Analyzing file ../data/invoice.pdf with analyzer: field-extraction-sample-df9a6007-a6a9-4b4c-999b-8f0d8b70298b
INFO:python.content_understanding_client:Request c3d358fe-9940-4cb5-a4fb-77f3c0571bca in progress ...
INFO:python.content_understanding_client:Request c3d358fe-9940-4cb5-a4fb-77f3c0571bca in progress ...
INFO:python.content_understanding_client:Request c3d358fe-9940-4cb5-a4fb-77f3c0571bca in progress ...
INFO:python.content_understanding_client:Request result is ready after 7.32 seconds.


{
  "id": "c3d358fe-9940-4cb5-a4fb-77f3c0571bca",
  "status": "Succeeded",
  "result": {
    "analyzerId": "field-extraction-sample-df9a6007-a6a9-4b4c-999b-8f0d8b70298b",
    "apiVersion": "2024-12-01-preview",
    "createdAt": "2025-03-06T17:00:53Z",
    "warnings": [],
    "contents": [
      {
        "markdown": "CONTOSO LTD.\n\n\n# INVOICE\n\nContoso Headquarters\n123 456th St\nNew York, NY, 10001\n\nINVOICE: INV-100\n\nINVOICE DATE: 11/15/2019\n\nDUE DATE: 12/15/2019\n\nCUSTOMER NAME: MICROSOFT CORPORATION\n\nSERVICE PERIOD: 10/14/2019 - 11/14/2019\n\nCUSTOMER ID: CID-12345\n\nMicrosoft Corp\n123 Other St,\nRedmond WA, 98052\n\nBILL TO:\n\nMicrosoft Finance\n\n123 Bill St,\n\nRedmond WA, 98052\n\nSHIP TO:\n\nMicrosoft Delivery\n\n123 Ship St,\n\nRedmond WA, 98052\n\nSERVICE ADDRESS:\nMicrosoft Services\n123 Service St,\nRedmond WA, 98052\n\n\n<table>\n<tr>\n<th>SALESPERSON</th>\n<th>P.O. NUMBER</th>\n<th>REQUISITIONER</th>\n<th>SHIPPED VIA</th>\n<th>F.O.B. POINT</th>\n<th>TERMS</

In [14]:
client.delete_analyzer(ANALYZER_ID)

INFO:python.content_understanding_client:Analyzer field-extraction-sample-df9a6007-a6a9-4b4c-999b-8f0d8b70298b deleted.


<Response [204]>

## Extract Fields Using the Analyzer

In [15]:
import uuid

ANALYZER_TEMPLATE = "chart"
ANALYZER_ID = "field-extraction-sample-" + str(uuid.uuid4())

(analyzer_template_path, analyzer_sample_file_path) = extraction_templates[ANALYZER_TEMPLATE]

In [16]:
response = client.begin_create_analyzer(ANALYZER_ID, analyzer_template_path=analyzer_template_path)
result = client.poll_result(response)

print(json.dumps(result, indent=2))

INFO:python.content_understanding_client:Analyzer field-extraction-sample-57762434-a872-47be-ab1b-57fa38be0bdf create request accepted.
INFO:python.content_understanding_client:Request result is ready after 0.00 seconds.


{
  "id": "7877d458-93ea-4ecd-a8d7-bc27580631ad",
  "status": "Succeeded",
  "result": {
    "analyzerId": "field-extraction-sample-57762434-a872-47be-ab1b-57fa38be0bdf",
    "description": "Extract detailed structured information from charts and diagrams.",
    "createdAt": "2025-03-06T17:01:02Z",
    "lastModifiedAt": "2025-03-06T17:01:02Z",
    "config": {
      "returnDetails": false,
      "disableContentFiltering": false
    },
    "fieldSchema": {
      "name": "ChartAndDiagram",
      "description": "Structured information from charts and diagrams.",
      "fields": {
        "Title": {
          "type": "string",
          "method": "generate",
          "description": "Verbatim title of the chart."
        },
        "ChartType": {
          "type": "string",
          "method": "classify",
          "description": "The type of chart.",
          "enum": [
            "area",
            "bar",
            "box",
            "bubble",
            "candlestick",
            "f

In [17]:
response = client.begin_analyze(ANALYZER_ID, file_location=analyzer_sample_file_path)
result = client.poll_result(response)

print(json.dumps(result, indent=2))

INFO:python.content_understanding_client:Analyzing file ../data/pieChart.jpg with analyzer: field-extraction-sample-57762434-a872-47be-ab1b-57fa38be0bdf
INFO:python.content_understanding_client:Request 1c6ad6b3-df37-4881-a4c4-8786a9610cdf in progress ...
INFO:python.content_understanding_client:Request 1c6ad6b3-df37-4881-a4c4-8786a9610cdf in progress ...
INFO:python.content_understanding_client:Request 1c6ad6b3-df37-4881-a4c4-8786a9610cdf in progress ...
INFO:python.content_understanding_client:Request 1c6ad6b3-df37-4881-a4c4-8786a9610cdf in progress ...
INFO:python.content_understanding_client:Request 1c6ad6b3-df37-4881-a4c4-8786a9610cdf in progress ...
INFO:python.content_understanding_client:Request 1c6ad6b3-df37-4881-a4c4-8786a9610cdf in progress ...
INFO:python.content_understanding_client:Request result is ready after 14.75 seconds.


{
  "id": "1c6ad6b3-df37-4881-a4c4-8786a9610cdf",
  "status": "Succeeded",
  "result": {
    "analyzerId": "field-extraction-sample-57762434-a872-47be-ab1b-57fa38be0bdf",
    "apiVersion": "2024-12-01-preview",
    "createdAt": "2025-03-06T17:01:03Z",
    "warnings": [],
    "contents": [
      {
        "markdown": "![image](image)\n",
        "fields": {
          "Title": {
            "type": "string",
            "valueString": "Pie Chart of Weekly Hours Worked"
          },
          "ChartType": {
            "type": "string",
            "valueString": "pie"
          },
          "TopicKeywords": {
            "type": "array",
            "valueArray": [
              {
                "type": "string",
                "valueString": "Employment"
              },
              {
                "type": "string",
                "valueString": "Time management"
              },
              {
                "type": "string",
                "valueString": "Work hours"
       

## Clean Up
Optionally, delete the sample analyzer from your resource. In typical usage scenarios, you would analyze multiple files using the same analyzer.

In [18]:
client.delete_analyzer(ANALYZER_ID)

INFO:python.content_understanding_client:Analyzer field-extraction-sample-57762434-a872-47be-ab1b-57fa38be0bdf deleted.


<Response [204]>